In [ ]:
%pip install flask flask-cors requests

In [24]:
from flask import Flask, request, jsonify
from flask_cors import CORS

from pathlib import Path

import getpass
import hashlib
import json
import math
import sqlite3
import threading
import time
import requests

from werkzeug.serving import make_server

In [43]:
# ============================================================
# CONFIGURATION
# ============================================================

MAPILLARY_TOKEN = getpass.getpass(
    "Paste your Mapillary access token: "
)

MAPILLARY_IMAGES_URL = (
    "https://graph.mapillary.com/images"
)

EARTH_RADIUS = 6371000


# ------------------------------------------------------------
# Locate project directory
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR


# ============================================================
# CACHE LOCATION
# ============================================================
#
# IMPORTANT:
# Keep runtime/database files OUTSIDE the website directory.
# This prevents file watchers / dev servers from treating
# backend writes as frontend source-code changes.
# ============================================================

CACHE_DIR = (
    Path.home()
    / ".young_family_access"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CACHE_DB = (
    CACHE_DIR
    / "journey_cache.sqlite"
)

CACHE_TTL_SECONDS = (
    7 * 24 * 60 * 60
)

print("Cache database:")
print(CACHE_DB)

CACHE_DB.parent.mkdir(
    parents=True,
    exist_ok=True
)


# Cache lifetime = 7 days

CACHE_TTL_SECONDS = (
    7 * 24 * 60 * 60
)


print("Project directory:")
print(PROJECT_DIR)

print()

print("Cache database:")
print(CACHE_DB)

print()

print(
    "Mapillary token loaded:",
    bool(MAPILLARY_TOKEN)
)

Paste your Mapillary access token:  ········


Cache database:
C:\Users\MSI\.young_family_access\journey_cache.sqlite
Project directory:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search

Cache database:
C:\Users\MSI\.young_family_access\journey_cache.sqlite

Mapillary token loaded: True


In [44]:
# ============================================================
# GEOGRAPHIC HELPERS
# ============================================================

def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2
):

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    d_phi = math.radians(
        lat2 - lat1
    )

    d_lambda = math.radians(
        lon2 - lon1
    )

    a = (
        math.sin(d_phi / 2) ** 2
        +
        math.cos(phi1)
        *
        math.cos(phi2)
        *
        math.sin(d_lambda / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return EARTH_RADIUS * c



def interpolate_point(
    start,
    end,
    fraction
):

    return {
        "lat":
            start["lat"]
            +
            (
                end["lat"]
                -
                start["lat"]
            )
            *
            fraction,

        "lon":
            start["lon"]
            +
            (
                end["lon"]
                -
                start["lon"]
            )
            *
            fraction
    }

In [45]:
# ============================================================
# ROUTE SAMPLING
# ============================================================

def sample_route(
    route_points,
    spacing=50
):

    if len(route_points) < 2:
        return []

    samples = [
        {
            "lat": route_points[0]["lat"],
            "lon": route_points[0]["lon"],
            "distanceFromStart": 0
        }
    ]

    total_distance = 0.0
    next_sample = float(spacing)

    for i in range(
        1,
        len(route_points)
    ):

        start = route_points[i - 1]
        end = route_points[i]

        segment_length = haversine_m(
            start["lat"],
            start["lon"],
            end["lat"],
            end["lon"]
        )

        if segment_length <= 0:
            continue

        previous_total = total_distance

        total_distance += segment_length

        while (
            next_sample
            <= total_distance
        ):

            distance_into_segment = (
                next_sample
                -
                previous_total
            )

            fraction = (
                distance_into_segment
                /
                segment_length
            )

            fraction = max(
                0.0,
                min(
                    1.0,
                    fraction
                )
            )

            point = interpolate_point(
                start,
                end,
                fraction
            )

            samples.append(
                {
                    "lat":
                        point["lat"],

                    "lon":
                        point["lon"],

                    "distanceFromStart":
                        round(next_sample)
                }
            )

            next_sample += spacing

    final = route_points[-1]

    distance_to_final = haversine_m(
        samples[-1]["lat"],
        samples[-1]["lon"],
        final["lat"],
        final["lon"]
    )

    if distance_to_final > 15:

        samples.append(
            {
                "lat":
                    final["lat"],

                "lon":
                    final["lon"],

                "distanceFromStart":
                    round(total_distance)
            }
        )

    return samples

In [46]:
# ============================================================
# SQLITE CACHE
# ============================================================

def get_db():

    connection = sqlite3.connect(
        CACHE_DB,
        timeout=30
    )

    connection.row_factory = (
        sqlite3.Row
    )

    return connection



def initialise_cache():

    with get_db() as db:

        db.execute("""
            CREATE TABLE IF NOT EXISTS route_cache (

                route_key TEXT PRIMARY KEY,

                response_json TEXT NOT NULL,

                created_at REAL NOT NULL
            )
        """)

        db.execute("""
            CREATE TABLE IF NOT EXISTS checkpoint_cache (

                checkpoint_key TEXT PRIMARY KEY,

                sample_lat REAL NOT NULL,
                sample_lon REAL NOT NULL,

                image_id TEXT,
                image_url TEXT,

                image_lat REAL,
                image_lon REAL,

                image_distance REAL,

                captured_at INTEGER,
                compass_angle REAL,

                created_at REAL NOT NULL
            )
        """)

        db.execute("""
            CREATE INDEX IF NOT EXISTS idx_checkpoint_location

            ON checkpoint_cache(
                sample_lat,
                sample_lon
            )
        """)

        db.commit()

    print(
        "SQLite cache ready."
    )


initialise_cache()

SQLite cache ready.


In [47]:
# ============================================================
# CACHE HELPERS
# ============================================================

def checkpoint_key(
    lat,
    lon
):

    return (
        f"{round(lat, 4)}_"
        f"{round(lon, 4)}"
    )



def route_key(
    route_points,
    spacing
):

    simplified = [
        [
            round(
                point["lat"],
                5
            ),
            round(
                point["lon"],
                5
            )
        ]
        for point
        in route_points
    ]

    payload = {
        "route": simplified,
        "spacing": spacing
    }

    encoded = json.dumps(
        payload,
        sort_keys=True
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        encoded
    ).hexdigest()



def cache_is_fresh(
    created_at
):

    age = (
        time.time()
        -
        created_at
    )

    return (
        age
        <
        CACHE_TTL_SECONDS
    )



def get_cached_route(
    key
):

    with get_db() as db:

        row = db.execute(
            """
            SELECT
                response_json,
                created_at

            FROM route_cache

            WHERE route_key = ?
            """,
            (
                key,
            )
        ).fetchone()

    if not row:
        return None

    if not cache_is_fresh(
        row["created_at"]
    ):
        return None

    try:

        return json.loads(
            row["response_json"]
        )

    except Exception:

        return None



def save_cached_route(
    key,
    response
):

    with get_db() as db:

        db.execute(
            """
            INSERT OR REPLACE INTO route_cache (

                route_key,
                response_json,
                created_at

            )

            VALUES (?, ?, ?)
            """,
            (
                key,
                json.dumps(response),
                time.time()
            )
        )

        db.commit()



def get_cached_checkpoint(
    lat,
    lon
):

    key = checkpoint_key(
        lat,
        lon
    )

    with get_db() as db:

        row = db.execute(
            """
            SELECT *

            FROM checkpoint_cache

            WHERE checkpoint_key = ?
            """,
            (
                key,
            )
        ).fetchone()

    if not row:
        return None

    if not cache_is_fresh(
        row["created_at"]
    ):
        return None

    return dict(row)



def save_checkpoint(
    sample,
    image_data
):

    key = checkpoint_key(
        sample["lat"],
        sample["lon"]
    )

    with get_db() as db:

        db.execute(
            """
            INSERT OR REPLACE INTO checkpoint_cache (

                checkpoint_key,

                sample_lat,
                sample_lon,

                image_id,
                image_url,

                image_lat,
                image_lon,

                image_distance,

                captured_at,
                compass_angle,

                created_at
            )

            VALUES (
                ?, ?, ?,
                ?, ?,
                ?, ?,
                ?,
                ?, ?,
                ?
            )
            """,
            (
                key,

                sample["lat"],
                sample["lon"],

                (
                    image_data.get("id")
                    if image_data
                    else None
                ),

                (
                    image_data.get("image")
                    if image_data
                    else None
                ),

                (
                    image_data.get("lat")
                    if image_data
                    else None
                ),

                (
                    image_data.get("lon")
                    if image_data
                    else None
                ),

                (
                    image_data.get(
                        "distanceFromSample"
                    )
                    if image_data
                    else None
                ),

                (
                    image_data.get(
                        "capturedAt"
                    )
                    if image_data
                    else None
                ),

                (
                    image_data.get(
                        "compassAngle"
                    )
                    if image_data
                    else None
                ),

                time.time()
            )
        )

        db.commit()

In [48]:
# ============================================================
# ROUTE BOUNDING BOX
# ============================================================

def build_bbox(
    points,
    buffer_m=120
):

    min_lat = min(
        point["lat"]
        for point in points
    )

    max_lat = max(
        point["lat"]
        for point in points
    )

    min_lon = min(
        point["lon"]
        for point in points
    )

    max_lon = max(
        point["lon"]
        for point in points
    )

    centre_lat = (
        min_lat + max_lat
    ) / 2

    lat_buffer = (
        buffer_m / 111320
    )

    lon_buffer = (
        buffer_m
        /
        (
            111320
            *
            max(
                math.cos(
                    math.radians(
                        centre_lat
                    )
                ),
                0.01
            )
        )
    )

    return (
        min_lon - lon_buffer,
        min_lat - lat_buffer,
        max_lon + lon_buffer,
        max_lat + lat_buffer
    )

In [49]:
# ============================================================
# BATCH MAPILLARY REQUEST
# ============================================================

def fetch_mapillary_images_for_route(
    points,
    buffer_m=120,
    limit=500
):

    if not points:
        return []

    bbox = build_bbox(
        points,
        buffer_m=buffer_m
    )

    bbox_string = ",".join(
        str(value)
        for value in bbox
    )

    params = {
        "access_token":
            MAPILLARY_TOKEN,

        "bbox":
            bbox_string,

        "limit":
            limit,

        "fields":
            (
                "id,"
                "thumb_1024_url,"
                "computed_geometry,"
                "captured_at,"
                "compass_angle"
            )
    }

    print(
        "Sending one Mapillary request "
        "for entire route..."
    )

    response = requests.get(
        MAPILLARY_IMAGES_URL,
        params=params,
        timeout=20
    )

    if response.status_code != 200:

        print(
            "Mapillary error:",
            response.status_code
        )

        print(
            response.text[:500]
        )

        return []

    payload = response.json()

    raw_images = payload.get(
        "data",
        []
    )

    images = []

    for image in raw_images:

        geometry = image.get(
            "computed_geometry"
        )

        if not geometry:
            continue

        coordinates = geometry.get(
            "coordinates"
        )

        if (
            not coordinates
            or len(coordinates) < 2
        ):
            continue

        image_lon = coordinates[0]
        image_lat = coordinates[1]

        image_url = image.get(
            "thumb_1024_url"
        )

        if not image_url:
            continue

        images.append(
            {
                "id":
                    image.get("id"),

                "image":
                    image_url,

                "lat":
                    image_lat,

                "lon":
                    image_lon,

                "capturedAt":
                    image.get(
                        "captured_at"
                    ),

                "compassAngle":
                    image.get(
                        "compass_angle"
                    )
            }
        )

    print(
        f"Mapillary returned "
        f"{len(images)} candidate images."
    )

    return images

In [50]:
# ============================================================
# LOCAL IMAGE MATCHING
# ============================================================

def find_best_image_locally(
    sample,
    images,
    used_image_ids=None,
    max_distance_m=120
):

    if used_image_ids is None:
        used_image_ids = set()

    candidates = []

    for image in images:

        distance = haversine_m(
            sample["lat"],
            sample["lon"],
            image["lat"],
            image["lon"]
        )

        if distance <= max_distance_m:

            candidates.append(
                (
                    distance,
                    image
                )
            )

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: item[0]
    )

    chosen = None

    for distance, image in candidates:

        if image["id"] not in used_image_ids:

            chosen = (
                distance,
                image
            )

            break

    if chosen is None:
        chosen = candidates[0]

    distance, image = chosen

    result = dict(image)

    result[
        "distanceFromSample"
    ] = round(
        distance,
        1
    )

    return result

In [51]:
# ============================================================
# OPTIMIZED ROUTE ANALYSIS
# ============================================================

def analyse_route_fast(
    route_points,
    spacing=50
):

    start_time = (
        time.perf_counter()
    )

    # --------------------------------------------------------
    # COMPLETE ROUTE CACHE
    # --------------------------------------------------------

    key = route_key(
        route_points,
        spacing
    )

    cached_route = (
        get_cached_route(
            key
        )
    )

    if cached_route:

        elapsed = (
            time.perf_counter()
            -
            start_time
        ) * 1000

        cached_route[
            "source"
        ] = "route-cache"

        cached_route[
            "executionMs"
        ] = round(
            elapsed,
            1
        )

        cached_route[
            "mapillaryRequests"
        ] = 0

        print(
            "COMPLETE ROUTE CACHE HIT"
        )

        return cached_route

    # --------------------------------------------------------
    # SAMPLE ROUTE
    # --------------------------------------------------------

    samples = sample_route(
        route_points,
        spacing=spacing
    )

    cached_results = {}

    missing_samples = []

    for index, sample in enumerate(
        samples
    ):

        cached = get_cached_checkpoint(
            sample["lat"],
            sample["lon"]
        )

        if cached:

            cached_results[
                index
            ] = cached

        else:

            missing_samples.append(
                (
                    index,
                    sample
                )
            )

    cache_hits = len(
        cached_results
    )

    print(
        f"Checkpoint cache: "
        f"{cache_hits}/{len(samples)} hits"
    )

    # --------------------------------------------------------
    # MAPILLARY QUERY
    # --------------------------------------------------------

    mapillary_images = []

    mapillary_requests = 0

    if missing_samples:

        query_points = [
            sample
            for _, sample
            in missing_samples
        ]

        mapillary_images = (
            fetch_mapillary_images_for_route(
                query_points,
                buffer_m=120,
                limit=500
            )
        )

        mapillary_requests = 1

    # --------------------------------------------------------
    # MATCH CHECKPOINTS
    # --------------------------------------------------------

    moments = []

    used_image_ids = set()

    for index, sample in enumerate(
        samples
    ):

        cached = cached_results.get(
            index
        )

        if cached:

            image_data = {
                "id":
                    cached[
                        "image_id"
                    ],

                "image":
                    cached[
                        "image_url"
                    ],

                "lat":
                    cached[
                        "image_lat"
                    ],

                "lon":
                    cached[
                        "image_lon"
                    ],

                "distanceFromSample":
                    cached[
                        "image_distance"
                    ],

                "capturedAt":
                    cached[
                        "captured_at"
                    ],

                "compassAngle":
                    cached[
                        "compass_angle"
                    ]
            }

        else:

            image_data = (
                find_best_image_locally(
                    sample,
                    mapillary_images,
                    used_image_ids,
                    max_distance_m=120
                )
            )

            save_checkpoint(
                sample,
                image_data
            )

        if (
            image_data
            and
            image_data.get("id")
        ):

            used_image_ids.add(
                image_data["id"]
            )

        moments.append(
            {
                "index":
                    index + 1,

                "lat":
                    sample["lat"],

                "lon":
                    sample["lon"],

                "distanceFromStart":
                    sample[
                        "distanceFromStart"
                    ],

                "image":
                    (
                        image_data.get(
                            "image"
                        )
                        if image_data
                        else None
                    ),

                "imageProvider":
                    (
                        "Mapillary"
                        if image_data
                        and image_data.get(
                            "image"
                        )
                        else None
                    ),

                "imageId":
                    (
                        image_data.get(
                            "id"
                        )
                        if image_data
                        else None
                    ),

                "imageLat":
                    (
                        image_data.get(
                            "lat"
                        )
                        if image_data
                        else None
                    ),

                "imageLon":
                    (
                        image_data.get(
                            "lon"
                        )
                        if image_data
                        else None
                    ),

                "imageDistance":
                    (
                        image_data.get(
                            "distanceFromSample"
                        )
                        if image_data
                        else None
                    ),

                "capturedAt":
                    (
                        image_data.get(
                            "capturedAt"
                        )
                        if image_data
                        else None
                    ),

                "compassAngle":
                    (
                        image_data.get(
                            "compassAngle"
                        )
                        if image_data
                        else None
                    )
            }
        )

    elapsed = (
        time.perf_counter()
        -
        start_time
    ) * 1000

    response = {
        "sampleCount":
            len(moments),

        "imagesFound":
            sum(
                1
                for moment in moments
                if moment["image"]
            ),

        "checkpointCacheHits":
            cache_hits,

        "mapillaryRequests":
            mapillary_requests,

        "source":
            "computed",

        "executionMs":
            round(
                elapsed,
                1
            ),

        "moments":
            moments
    }

    save_cached_route(
        key,
        response
    )

    return response

In [52]:
# ============================================================
# LOCAL TEST
# ============================================================

test_route = [
    {
        "lat": 1.2995,
        "lon": 103.8452
    },
    {
        "lat": 1.2997,
        "lon": 103.8448
    },
    {
        "lat": 1.3002,
        "lon": 103.8443
    },
    {
        "lat": 1.3008,
        "lon": 103.8440
    }
]


test_result = analyse_route_fast(
    test_route,
    spacing=50
)


print()
print(
    "Sample count:",
    test_result["sampleCount"]
)

print(
    "Images found:",
    test_result["imagesFound"]
)

print(
    "Mapillary requests:",
    test_result["mapillaryRequests"]
)

print(
    "Execution:",
    test_result["executionMs"],
    "ms"
)

Checkpoint cache: 0/5 hits
Sending one Mapillary request for entire route...
Mapillary returned 210 candidate images.

Sample count: 5
Images found: 5
Mapillary requests: 1
Execution: 4489.2 ms


In [53]:
# ============================================================
# IMAGE CHECK
# ============================================================

for moment in test_result["moments"]:

    print(
        f"Moment {moment['index']} | "
        f"{moment['distanceFromStart']} m | "
        f"Image: {bool(moment['image'])} | "
        f"Distance: {moment['imageDistance']}"
    )

Moment 1 | 0 m | Image: True | Distance: 8.2
Moment 2 | 50 m | Image: True | Distance: 20.4
Moment 3 | 100 m | Image: True | Distance: 54.8
Moment 4 | 150 m | Image: True | Distance: 69.9
Moment 5 | 200 m | Image: True | Distance: 102.1


In [54]:
# ============================================================
# FLASK APP
# ============================================================

app = Flask(
    __name__
)

CORS(
    app,
    resources={
        r"/*": {
            "origins": "*"
        }
    }
)

In [55]:
# ============================================================
# API ENDPOINTS
# ============================================================

@app.route(
    "/",
    methods=["GET"]
)
def home():

    return jsonify(
        {
            "status":
                "running",

            "service":
                "young-family-route-analysis",

            "health":
                "/health",

            "analyseRoute":
                "/analyse-route",

            "cacheStats":
                "/cache-stats"
        }
    )



@app.route(
    "/health",
    methods=["GET"]
)
def health():

    return jsonify(
        {
            "status":
                "ok",

            "service":
                "young-family-route-analysis",

            "mapillaryConnected":
                bool(
                    MAPILLARY_TOKEN
                )
        }
    )



@app.route(
    "/analyse-route",
    methods=["POST"]
)
def analyse_route_endpoint():

    data = request.get_json(
        silent=True
    )

    if not data:

        return jsonify(
            {
                "error":
                    "No JSON supplied."
            }
        ), 400

    route_points = data.get(
        "route",
        []
    )

    spacing = int(
        data.get(
            "spacing",
            50
        )
    )

    if len(route_points) < 2:

        return jsonify(
            {
                "error":
                    "Route requires at least two points."
            }
        ), 400

    try:

        result = analyse_route_fast(
            route_points,
            spacing=spacing
        )

        return jsonify(
            result
        )

    except Exception as error:

        print(
            "Route analysis error:",
            error
        )

        return jsonify(
            {
                "error":
                    str(error)
            }
        ), 500

In [56]:
# ============================================================
# CACHE STATS
# ============================================================

@app.route(
    "/cache-stats",
    methods=["GET"]
)
def cache_stats():

    with get_db() as db:

        route_count = db.execute(
            """
            SELECT COUNT(*)
            FROM route_cache
            """
        ).fetchone()[0]

        checkpoint_count = db.execute(
            """
            SELECT COUNT(*)
            FROM checkpoint_cache
            """
        ).fetchone()[0]

        image_count = db.execute(
            """
            SELECT COUNT(*)
            FROM checkpoint_cache
            WHERE image_url IS NOT NULL
            """
        ).fetchone()[0]

    return jsonify(
        {
            "cachedRoutes":
                route_count,

            "cachedCheckpoints":
                checkpoint_count,

            "cachedImages":
                image_count
        }
    )

In [57]:
# ============================================================
# SERVER THREAD
# ============================================================

class ServerThread(
    threading.Thread
):

    def __init__(
        self,
        app,
        host="127.0.0.1",
        port=5000
    ):

        super().__init__()

        self.host = host
        self.port = port

        self.server = make_server(
            host,
            port,
            app,
            threaded=True
        )

        self.ctx = app.app_context()

        self.ctx.push()


    def run(self):

        print(
            "===================================="
        )

        print(
            "Young Family Route Backend"
        )

        print(
            "===================================="
        )

        print(
            f"Server: "
            f"http://{self.host}:{self.port}"
        )

        print(
            f"Health: "
            f"http://{self.host}:{self.port}/health"
        )

        print(
            f"Route API: "
            f"http://{self.host}:{self.port}/analyse-route"
        )

        print(
            f"Cache: "
            f"http://{self.host}:{self.port}/cache-stats"
        )

        print(
            "===================================="
        )

        self.server.serve_forever()


    def shutdown(self):

        self.server.shutdown()

In [58]:
# ============================================================
# START SERVER
# ============================================================

server = ServerThread(
    app
)

server.daemon = True

server.start()

Young Family Route Backend
Server: http://127.0.0.1:5000
Health: http://127.0.0.1:5000/health
Route API: http://127.0.0.1:5000/analyse-route
Cache: http://127.0.0.1:5000/cache-stats


In [59]:
# ============================================================
# HEALTH TEST
# ============================================================

health_test = requests.get(
    "http://127.0.0.1:5000/health"
)

print(
    health_test.status_code
)

print(
    health_test.json()
)

127.0.0.1 - - [12/Sep/2026 17:44:01] "GET /health HTTP/1.1" 200 -


200
{'mapillaryConnected': True, 'service': 'young-family-route-analysis', 'status': 'ok'}


In [60]:
# ============================================================
# API TEST
# ============================================================

api_test = requests.post(
    "http://127.0.0.1:5000/analyse-route",
    json={
        "route":
            test_route,

        "spacing":
            50
    }
)

print(
    "HTTP:",
    api_test.status_code
)

api_result = api_test.json()

print(
    "Source:",
    api_result.get(
        "source"
    )
)

print(
    "Samples:",
    api_result.get(
        "sampleCount"
    )
)

print(
    "Images:",
    api_result.get(
        "imagesFound"
    )
)

print(
    "Mapillary requests:",
    api_result.get(
        "mapillaryRequests"
    )
)

print(
    "Execution:",
    api_result.get(
        "executionMs"
    ),
    "ms"
)

127.0.0.1 - - [12/Sep/2026 17:44:01] "POST /analyse-route HTTP/1.1" 200 -


COMPLETE ROUTE CACHE HIT
HTTP: 200
Source: route-cache
Samples: 5
Images: 5
Mapillary requests: 0
Execution: 2.4 ms


127.0.0.1 - - [12/Sep/2026 17:44:08] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:44:08] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:44:08] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:44:08] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:44:08] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:44:08] "OPTIONS /analyse-route HTTP/1.1" 200 -


Checkpoint cache: 0/8 hits
Sending one Mapillary request for entire route...
Checkpoint cache: 0/10 hits
Sending one Mapillary request for entire route...
Checkpoint cache: 0/10 hits
Sending one Mapillary request for entire route...


127.0.0.1 - - [12/Sep/2026 17:44:13] "POST /analyse-route HTTP/1.1" 200 -


...

127.0.0.1 - - [12/Sep/2026 17:45:16] "POST /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:16] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:16] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:16] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:16] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:16] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:17] "POST /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:17] "POST /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:46:17] "POST /analyse-route HTTP/1.1" 200 -


COMPLETE ROUTE CACHE HIT
COMPLETE ROUTE CACHE HIT
COMPLETE ROUTE CACHE HIT


127.0.0.1 - - [12/Sep/2026 17:58:12] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:12] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:12] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:12] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:12] "OPTIONS /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:12] "OPTIONS /analyse-route HTTP/1.1" 200 -


Checkpoint cache: 0/8 hits
Sending one Mapillary request for entire route...
Checkpoint cache: 0/8 hits
Sending one Mapillary request for entire route...
Checkpoint cache: 0/7 hits
Sending one Mapillary request for entire route...
Mapillary returned 379 candidate images.
Mapillary returned 450 candidate images.
Mapillary returned 450 candidate images.


127.0.0.1 - - [12/Sep/2026 17:58:16] "POST /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:17] "POST /analyse-route HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 17:58:17] "POST /analyse-route HTTP/1.1" 200 -
